# Perfil de la variable objetivo (clasificación BTI)

Calcula lo que pidió el docente: **total de registros, número de categorías BTI, distribución y ejemplos por clase**. Abre y *Entorno de ejecución → Ejecutar todo*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, sqlite3, pandas as pd

# Ruta directa a la base (carpeta 'Especializacion IA'); si no esta, busca de forma robusta.
RUTA_DB = '/content/drive/MyDrive/Especializacion IA/incidentes_anonimizado.db'
if not os.path.exists(RUTA_DB):
    validos = [c for c in glob.glob('/content/drive/MyDrive/**/incidentes_anonimizado.db', recursive=True) if os.path.isfile(c)]
    if validos:
        RUTA_DB = max(validos, key=os.path.getsize)
print('USANDO:', RUTA_DB, '|', round(os.path.getsize(RUTA_DB)/1e6, 1), 'MB')

In [ ]:
con = sqlite3.connect(RUTA_DB)
ing = pd.read_sql('SELECT * FROM "ingresadas"', con); ing.columns = ing.columns.str.lower().str.strip()
COL = 'bti_clasificacion'   # columna del target
s = ing[COL].fillna('SIN_CLASIFICAR').value_counts()

print('1) Total de registros con BTI:', len(ing))
print('2) Categorias BTI distintas :', ing[COL].nunique())
print('\n3-4) Distribucion / ejemplos por categoria:')
print(s.to_string())
print('\nClases con < 10 ejemplos:', int((s < 10).sum()), 'de', len(s))
print('Las 10 mas frecuentes concentran:', round(s.head(10).sum()/s.sum()*100, 1), '% de los casos')

tk = next((c for c in ['ticket','llave','clave'] if c in ing.columns), None)
if tk:
    ix = ing[ing[tk].astype(str).str.upper().str.startswith('ITHD')]
    print('\n[IX Comercio] registros:', len(ix), '| categorias BTI:', ix[COL].nunique())

s.rename_axis('categoria_bti').reset_index(name='casos').to_csv('perfil_bti.csv', index=False, encoding='utf-8')
print('\nGuardado: perfil_bti.csv')

In [ ]:
import matplotlib.pyplot as plt
s.head(15)[::-1].plot(kind='barh', figsize=(8,6))
plt.title('Top 15 categorias BTI por número de casos'); plt.xlabel('Casos'); plt.tight_layout(); plt.show()